In [1]:
# pip install nba_api
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nba_api.stats.endpoints import playercareerstats, shotchartdetail, commonplayerinfo
from nba_api.stats.static import players
import time
import plotly.express as px
# import 


In [15]:
# all_players = players.get_players()
# sampled_players = np.random.choice(all_players, 500, replace=False)
# results = {'Name': [], 'ID': [], 'Height': [], 'Weight': [], 'Position': []}

# from nba_api.stats.library.parameters import SeasonAll
# from requests.exceptions import ReadTimeout

# for p in sampled_players:
#     try:
#         info = commonplayerinfo.CommonPlayerInfo(player_id=p['id'], timeout=5)  # 5 sec timeout
#         df = info.get_data_frames()[0]
#         results['Name'].append(p['full_name'])
#         results['ID'].append(p['id'])
#         results['Height'].append(df['HEIGHT'].values[0])
#         results['Weight'].append(df['WEIGHT'].values[0])
#         results['Position'].append(df['POSITION'].values[0])
#         time.sleep(0.5)
#     except ReadTimeout:
#         print(f"Timed out for {p['full_name']}, skipping.")
#         continue
#     except Exception as e:
#         print(f"Failed for {p['full_name']} - {e}")
#         continue

# df = pd.DataFrame(results).dropna()

# df['Height_inches'] = df['Height'].apply(lambda x: int(x.split('-')[0]) * 12 + int(x.split('-')[1]))

# position_map = {
#     'Guard': 'Guard',
#     'Forward': 'Forward',
#     'Center': 'Center',
#     'Guard-Forward': 'Guard',
#     'Forward-Guard': 'Guard',
#     'Forward-Center': 'Forward',
#     'Center-Forward': 'Center'
# }

# df['Archetype'] = df['Position'].map(position_map)

# df.to_csv('../../data/player_info.csv', index=False)


In [4]:
df = pd.read_csv('../../data/player_info.csv')
df

,Name,ID,Height,Weight,Position,Height_inches,Archetype
0,Amir Coffey,1629599,6-7,210.0,Guard-Forward,79,Guard
1,Evan Turner,202323,6-6,220.0,Guard-Forward,78,Guard
2,Derek Hood,1967,6-8,222.0,Forward,80,Forward
3,Jack Eskridge,76682,6-5,200.0,Center,77,Center
4,Nate Hawthorne,76978,6-4,190.0,Guard,76,Guard
...,...,...,...,...,...,...,...
483,Ryan Lorthridge,943,6-4,190.0,Guard,76,Guard
484,Henry Pearcy,77826,6-1,170.0,Guard,73,Guard
485,Bryce Cotton,203955,6-1,165.0,Guard,73,Guard
486,Bob Dandridge,76500,6-6,195.0,Forward,78,Forward


In [5]:
position_map = {
    'Guard': 'Guard',
    'Forward': 'Forward',
    'Center': 'Center',
    'Guard-Forward': 'Guard',
    'Forward-Guard': 'Guard',
    'Forward-Center': 'Forward',
    'Center-Forward': 'Forward'
}

df['Archetype'] = df['Position'].map(position_map)

In [6]:
df

,Name,ID,Height,Weight,Position,Height_inches,Archetype
0,Amir Coffey,1629599,6-7,210.0,Guard-Forward,79,Guard
1,Evan Turner,202323,6-6,220.0,Guard-Forward,78,Guard
2,Derek Hood,1967,6-8,222.0,Forward,80,Forward
3,Jack Eskridge,76682,6-5,200.0,Center,77,Center
4,Nate Hawthorne,76978,6-4,190.0,Guard,76,Guard
...,...,...,...,...,...,...,...
483,Ryan Lorthridge,943,6-4,190.0,Guard,76,Guard
484,Henry Pearcy,77826,6-1,170.0,Guard,73,Guard
485,Bryce Cotton,203955,6-1,165.0,Guard,73,Guard
486,Bob Dandridge,76500,6-6,195.0,Forward,78,Forward


In [7]:
df['Archetype'].value_counts()

Archetype
Guard      221
Forward    216
Center      51
Name: count, dtype: int64

In [8]:
fig = px.scatter(df, x='Height_inches', y='Weight', color='Archetype')
fig.show()
fig = px.scatter(df, x='Height_inches', y='Weight', color='Position')
fig.show()

In [9]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

In [65]:
X = df[['Height_inches', 'Weight']]
y = df['Archetype']

In [75]:
le = LabelEncoder()
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
y_encoded = le.fit_transform(y)

In [76]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2
)

In [77]:
param_grid = {
    'n_estimators': [15, 20, 25, 35, 40],
    'max_depth': [None, 2, 5, 7, 10],
    'min_samples_split': [15, 17, 20, 25]
}

grid = GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
    verbose=1,
    refit=True,
    return_train_score=True
)

grid.fit(X_train, y_train)

print(grid.best_params_)
print()
print(grid.score(X_test, y_test))

Fitting 5 folds for each of 100 candidates, totalling 500 fits
{'max_depth': 7, 'min_samples_split': 17, 'n_estimators': 20}

0.7653061224489796


In [109]:
def predict_player_type(height, weight, is_male = True):
    avg_male_height, avg_male_weight = 69.1, 199.8
    avg_female_height, avg_female_weight = 63.7, 170.8

    std_male_height, std_male_weight = 2.9, 40.8
    std_female_height, std_female_weight = 2.7, 40.5

    if is_male:
        height_scaled = (height - avg_male_height) / std_male_height
        weight_scaled = (weight - avg_male_weight) / std_male_weight
    else:
        height_scaled = (height - avg_female_height) / std_female_height
        weight_scaled = (weight - avg_female_weight) / std_female_weight

    user_scaled = np.array([[height_scaled, weight_scaled]])
    encoded_pred = grid.predict(user_scaled)
    decoded_pred = le.inverse_transform(encoded_pred)[0]
    return decoded_pred

predict_player_type(76, 100, False)

'Forward'